<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/Phase_11_K_Nearest_Neighbour/11_kNN_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 11: Classification via k-Nearest Neighbors (k-NN)
## Step 1: Loading the Compressed Data & Splitting the Environment

In Phase 10, we aggressively reduced our 590-sensor manufacturing dataset down to 162 mathematically independent components, preserving 95% of the vital business signal while eliminating the "Curse of Dimensionality."

Our first step is to load this optimized artifact (`secom_pca_95variance_ready.csv`) and split it into Training and Testing sets. Because predictive maintenance datasets are notoriously imbalanced (factory machines pass inspection far more often than they fail), we must use a **Stratified Split**. This guarantees that the rare anomalies are distributed evenly between our training and testing environments, preventing the model from learning a skewed representation of reality.

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("⏳ Loading PCA-compressed factory data...")

# 1. Load the artifact exported from Phase 10
dataset_path = 'secom_pca_95variance_ready.csv'
df = pd.read_csv(dataset_path)

# 2. Separate the Principal Components from the Target Label
X = df.drop(columns=['Target'])
y = df['Target']

# 3. Execute a Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Verify the distribution of anomalies
print("✅ Data Loaded and Split Successfully!\n")
print("📊 Training Environment:")
print(f"Total Batches: {len(y_train)}")
print(f"Normal Passes (-1): {sum(y_train == -1)}")
print(f"Anomalies / Fails (1): {sum(y_train == 1)}")

print("\n🧪 Testing Environment:")
print(f"Total Batches: {len(y_test)}")
print(f"Normal Passes (-1): {sum(y_test == -1)}")
print(f"Anomalies / Fails (1): {sum(y_test == 1)}")

⏳ Loading PCA-compressed factory data...


FileNotFoundError: [Errno 2] No such file or directory: 'secom_pca_95variance_ready.csv'

## Step 2: Instantiating and Training the k-NN Engine

With our anomalies perfectly stratified, we can build the k-Nearest Neighbors classifier.

Because k-NN is purely distance-based and we have already mathematically condensed our features using PCA, this model will be incredibly lightweight and fast. This minimal computational footprint is exactly what you want when packaging a predictive engine to run efficiently as the backend for an iOS or Android app later.

We will start by setting `n_neighbors=5`. This means for every new batch, the algorithm will look at the 5 closest historical batches in our 162-dimensional space and take a majority vote to decide if the new batch is a Pass or a Fail.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import time

print("⚙️ Initializing k-NN Engine (k=5)...")

# 1. Initialize the classifier with 5 neighbors
knn_baseline = KNeighborsClassifier(n_neighbors=5)

# 2. Train the model and track the execution time
start_time = time.time()
knn_baseline.fit(X_train, y_train)
training_time = time.time() - start_time

# 3. Generate predictions for the unseen test environment
y_pred = knn_baseline.predict(X_test)

print(f"✅ Model trained in {training_time:.4f} seconds.")
print("🔮 Predictions generated for the 314 test batches.")

# Step 3: Evaluating the Baseline Model (The Accuracy Trap)

In manufacturing anomaly detection, overall "Accuracy" is a dangerous metric. Because over 93% of our testing data consists of normal passes (293 out of 314), a completely broken model that blindly predicts "Pass" every single time would still technically score 93% accuracy, while missing 100% of the actual machine failures.

To see the true business value of our model, we must evaluate it using a **Confusion Matrix** and a **Classification Report**. We need to look specifically at the minority class (the anomalies) to check our **Recall** (out of all actual failures, how many did we catch?) and **Precision** (when we predicted a failure, how often were we right?).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("📊 Evaluating Baseline k-NN Performance...\n")

# 1. Print the strict Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Pass (-1)", "Fail (1)"], zero_division=0))

# 2. Generate the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# 3. Plot the matrix with a clean corporate aesthetic
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(6, 5), facecolor='#161b22')
ax.set_facecolor('#161b22')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=["Predicted Pass", "Predicted Fail"],
            yticklabels=["Actual Pass", "Actual Fail"],
            annot_kws={"size": 16, "weight": "bold"}, ax=ax)

ax.set_title('k-NN Confusion Matrix (k=5)', color='white', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(colors='#8b949e', labelsize=11)
plt.tight_layout()
plt.show()

# Detailed Discussion:
Overall accuracy is a misleading metric here. The model has 94% accuracy (295 out of 314 correct), which looks incredible on a high-level report.

But in a predictive maintenance scenario, this model is a disaster:

* Actual Machine Failures: 21 batches failed QC.

* Detected by the Model: 2.

* Missed (False Negatives): 19.

The current algorithm is too lazy. Because the 1170 "Pass" points dominate the space, a simple majority vote of the 5 closest neighbors almost always picks "Pass." This model has a terrible 10% Recall for the anomaly class. It only catches a failure if it's practically sitting on top of another one.

To solve this, we can't manually guess at the number of neighbors. We need to systematically sweep through different settings to force the algorithm to prioritize the minority class (the anomalies).

## Step 4: Hyperparameter Tuning via Grid Search

The baseline model (k=5) is biased toward the majority class, achieving high overall accuracy but missing 19 out of 21 machine failures (90% failure rate). To fix this, we need to mathematically force the k-NN model to be more sensitive to anomalies.

Instead of guessing, we will execute a **Grid Search with Cross-Validation**. This script acts as an automated scientist, systematically creating and testing dozens of different model combinations to find the optimal settings.

We will test:
1.  **n_neighbors (k):** The number of neighboring points to include in the vote (we'll sweep from 1 to 30).
2.  **weights:** 'uniform' (everyone gets an equal vote) vs. 'distance' (closer neighbors get a much stronger vote, which helps the minority class in crowded spaces).
3.  **metric:** 'Euclidean' (standard straight-line distance) vs. 'Manhattan' (distance along grid-like axes).

The Grid Search won't optimize for misleading "Accuracy." We will configure it to strictly maximize the **f1_score for the "Fail" class (Target=1)**, forcing it to prioritize detecting the rare anomalies.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

print("⏳ Executing Hyperparameter Grid Search (this may take a minute)...")

# 1. Define the grid of parameters to test (the automated search space)
param_grid = {
    'n_neighbors': list(range(1, 31, 2)), # Testing odd k values from 1 to 29
    'weights': ['uniform', 'distance'],  # Test uniform vs. distance-based voting
    'metric': ['euclidean', 'manhattan'] # Test straight-line vs. grid-like distance
}

# 2. Define a strict scoring function to maximize detection of the anomalies
anomaly_scorer = make_scorer(f1_score, pos_label=1)

# 3. Initialize the Grid Search with the scorer and 5-fold stratified cross-validation
grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring=anomaly_scorer,
    cv=5,                 # Validate 5 times to prevent overfitting to a single split
    n_jobs=-1,            # Use all available CPU cores for speed
    verbose=1
)

# 4. Execute the automated search on the training data
grid_search.fit(X_train, y_train)

print(f"✅ Grid Search Complete!\n")

# 5. Extract and display the optimal settings found
print("📊 Best Model Settings found for Anomaly Detection:")
print(f"K-Neighbors (k): {grid_search.best_params_['n_neighbors']}")
print(f"Distance Metric: {grid_search.best_params_['metric']}")
print(f"Voting System:  {grid_search.best_params_['weights']}")
print(f"Best Validation F1-Score: {grid_search.best_score_:.4f}")

#Explanation:
Selecting k=1 is a massive red flag in machine learning. It means the model is giving up on finding a broader pattern and is instead choosing to simply memorize the training data. It is looking at a new factory batch and blindly copying the status of the single closest historical data point. While this technically maximizes the score on paper, it creates a brittle, hyper-sensitive boundary. When packaging this predictive engine to run smoothly as a real-time mobile application backend, a model that memorizes rather than generalizes will fail entirely the moment it encounters slightly variant real-world data.

The F1-Score of 0.1777 confirms this instability. The F1-Score is the harmonic mean of Precision and Recall. A score of 0.17 means that even with the algorithm actively trying its hardest to find the anomalies, it is still missing the vast majority of them, and when it does guess "Fail," it is frequently wrong.


With 1,170 Normal Passes suffocating just 83 Failures in the training environment, the anomaly points are essentially isolated islands in a massive sea of normal operations. Because k-NN relies purely on local neighborhood density to cast its votes, the anomalies simply do not have enough density to win a vote unless k=1 (where only the single closest point matters).

To fix this, we have to intervene at the data level before we train the engine. The industry standard for this exact problem is SMOTE (Synthetic Minority Over-sampling Technique). Instead of just duplicating the 83 failures, SMOTE uses underlying geometry to synthetically generate brand new, mathematically plausible anomaly data points. It builds a bridge between those isolated islands until the training environment has an equal 50/50 split of passes and failures, forcing the model to actually learn the anomaly patterns.

## Step 1.5: Restoring Balance with SMOTE

Our initial data split revealed a severe class imbalance: 1,170 Normal Passes to only 83 Anomalies. If we feed this directly into a distance-based algorithm like k-NN, the sheer density of the "Pass" data points will suffocate the minority class, causing the model to default to predicting "Pass" almost every time.

To fix the root cause, we apply **SMOTE (Synthetic Minority Over-sampling Technique)** strictly to our Training Data. SMOTE analyzes the geometric distance between our 83 existing anomalies and mathematically synthesizes brand new, highly realistic anomaly data points. We will generate enough synthetic failures to bring the training environment to a perfect 1:1 balance, forcing the k-NN model to respect the anomaly boundary.

*Note: We never apply SMOTE to the Test Data. The Test Data must remain a strict, unbalanced representation of the real world.*

In [ ]:
from imblearn.over_sampling import SMOTE

print("⚖️ Synthesizing minority class data with SMOTE...\n")

# 1. Initialize SMOTE
smote = SMOTE(random_state=42)

# 2. Fit and resample ONLY on the training data
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# 3. Verify the new structural balance
print("📊 Original Training Environment:")
print(f"Normal Passes (-1): {sum(y_train == -1)}")
print(f"Anomalies / Fails (1): {sum(y_train == 1)}")

print("\n⚖️ New Balanced Training Environment:")
print(f"Normal Passes (-1): {sum(y_train_balanced == -1)}")
print(f"Anomalies / Fails (1): {sum(y_train_balanced == 1)}")

In [ ]:
# Re-training the Baseline k-NN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("⚙️ Re-training k-NN Engine on Balanced Data (k=5)...\n")

# 1. Train on the newly balanced SMOTE data
knn_balanced = KNeighborsClassifier(n_neighbors=5)
knn_balanced.fit(X_train_balanced, y_train_balanced)

# 2. Predict on the original, untouched Test Data
y_pred_balanced = knn_balanced.predict(X_test)

# 3. Print the strict Classification Report
print("Classification Report (Post-SMOTE):")
print(classification_report(y_test, y_pred_balanced, target_names=["Pass (-1)", "Fail (1)"], zero_division=0))

# 4. Generate and plot the new Confusion Matrix
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(6, 5), facecolor='#161b22')
ax.set_facecolor('#161b22')

sns.heatmap(cm_balanced, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=["Predicted Pass", "Predicted Fail"],
            yticklabels=["Actual Pass", "Actual Fail"],
            annot_kws={"size": 16, "weight": "bold"}, ax=ax)

ax.set_title('k-NN Confusion Matrix (Balanced Data)', color='white', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(colors='#8b949e', labelsize=11)
plt.tight_layout()
plt.show()

#Explanation
SMOTE did exactly what we engineered it to do: it forced the model to stop ignoring the anomalies.

Looking at your new Confusion Matrix, the impact on our Recall is massive. Before SMOTE, the model only caught 2 actual machine failures. Now, it has successfully hunted down and flagged 15 out of the 21 actual failures (a 71% Recall rate).

However, we have introduced a severe new business problem: Alarm Fatigue.
Because we forced the algorithm to be hyper-sensitive to failures, it has become utterly paranoid. It threw 193 false alarms (predicting a failure when the machine was actually running perfectly). If you deploy a predictive engine with a Precision of 7%, the factory floor operators will quickly start ignoring the alerts.

This is exactly why we perform Hyperparameter Tuning after fixing the data distribution. We will now run our Grid Search again, but this time on the X_train_balanced data, to find the mathematically optimal number of neighbors (k) that tightens up those false alarms without losing our high detection rate.

## Step 5: Hyperparameter Tuning the Balanced Engine

With our training data perfectly balanced via SMOTE, our baseline k-NN model achieved a 71% Recall (catching 15 of 21 failures), but at the cost of 193 false alarms.

To make this model production-ready, we must tighten its precision. We will execute a Grid Search strictly on the balanced training data to find the optimal geometric settings (`n_neighbors`, `weights`, `metric`) that maximize the F1-Score, finding the best mathematical compromise between catching failures and minimizing false alarms.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

print("⏳ Executing Grid Search on SMOTE-Balanced Data...")

# 1. Define the search space
param_grid = {
    'n_neighbors': list(range(1, 31, 2)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 2. Score strictly on the minority class F1-Score
anomaly_scorer = make_scorer(f1_score, pos_label=1)

# 3. Initialize Grid Search
grid_search_balanced = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring=anomaly_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1
)

# 4. Execute on the BALANCED training data
grid_search_balanced.fit(X_train_balanced, y_train_balanced)

print(f"✅ Balanced Grid Search Complete!\n")

print("📊 Optimal Production Settings:")
print(f"K-Neighbors (k): {grid_search_balanced.best_params_['n_neighbors']}")
print(f"Distance Metric: {grid_search_balanced.best_params_['metric']}")
print(f"Voting System:  {grid_search_balanced.best_params_['weights']}")

# 5. Lock in the best model
best_knn = grid_search_balanced.best_estimator_

# 6. Final Evaluation on the untouched Test Data
y_pred_final = best_knn.predict(X_test)

print("\n🏆 Final Production Classification Report:")
print(classification_report(y_test, y_pred_final, target_names=["Pass (-1)", "Fail (1)"], zero_division=0))